# 03 — Merge & Prepare ML Dataset

**Goal:** Load the cleaned historical panel, define features/targets, split into train/test.  
**Reads:** `data/processed/historical_panel.csv`  
**Outputs:** `data/processed/train.csv`, `data/processed/test.csv`

Features (5): `gdp_per_capita`, `hdi`, `control_of_corruption`, `employment_agriculture`, `gini`  
Targets (3): `poverty_3`, `poverty_8_30`, `poverty_10`  
Primary target: `poverty_3`

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.

## Imports and paths

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FEATURES = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
TARGETS = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)
PRIMARY_TARGET = "poverty_3"

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Targets  ({len(TARGETS)}): {TARGETS}")
print(f"Primary target: {PRIMARY_TARGET}")

## Load historical panel

In [ ]:
panel = pd.read_csv(DATA_PROCESSED / "historical_panel.csv")
print(f"Raw panel: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}, Years: {panel['year'].min()}–{panel['year'].max()}")
panel.head()

In [ ]:
# Quick look at missing data before filtering
print("% NaN per column:")
print((panel[FEATURES + TARGETS].isna().mean() * 100).round(2))

## Filter rows

- Drop rows where **all** target columns are NaN (no poverty data at all)
- Drop rows where **all** feature columns are NaN

In [ ]:
n_before = len(panel)

# Drop rows with ALL targets NaN
panel = panel.dropna(subset=TARGETS, how="all")
print(f"After dropping all-NaN targets: {len(panel)} (removed {n_before - len(panel)})")

# Drop rows with ALL features NaN
n_before2 = len(panel)
panel = panel.dropna(subset=FEATURES, how="all")
print(f"After dropping all-NaN features: {len(panel)} (removed {n_before2 - len(panel)})")

print(f"\nFinal dataset: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}")

### Check primary target coverage

In [ ]:
for t in TARGETS:
    n_valid = panel[t].notna().sum()
    print(f"{t}: {n_valid} non-null rows ({n_valid/len(panel)*100:.1f}%)")

print(f"\nRows with valid {PRIMARY_TARGET}: {panel[PRIMARY_TARGET].notna().sum()}")

## Train / Test split

80/20 random split, stratified by `country_code` so each country appears in both sets when possible.  
We use `random_state=42` for reproducibility.

In [ ]:
# Use GroupShuffleSplit to keep country groups partially in both splits
# Simpler approach: stratify by country_code using train_test_split
# Since some countries have very few rows, we group rare countries together

# Count rows per country
country_counts = panel["country_code"].value_counts()
print(f"Countries with 1 row: {(country_counts == 1).sum()}")
print(f"Countries with 2+ rows: {(country_counts >= 2).sum()}")

# For stratification: countries with <2 rows can't be stratified,
# so we create a stratification key that groups rare countries together
rare_countries = set(country_counts[country_counts < 2].index)
panel["strat_key"] = panel["country_code"].apply(
    lambda x: "_RARE_" if x in rare_countries else x
)

train, test = train_test_split(
    panel,
    test_size=0.2,
    random_state=42,
    stratify=panel["strat_key"]
)

# Drop the helper column
train = train.drop(columns=["strat_key"]).reset_index(drop=True)
test = test.drop(columns=["strat_key"]).reset_index(drop=True)

print(f"\nTrain: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Train countries: {train['country_code'].nunique()}")
print(f"Test countries:  {test['country_code'].nunique()}")

### Verify the split looks reasonable

In [ ]:
print("=== Train set ===")
print(f"Year range: {train['year'].min()}–{train['year'].max()}")
print(f"{PRIMARY_TARGET} — mean: {train[PRIMARY_TARGET].mean():.2f}, "
      f"median: {train[PRIMARY_TARGET].median():.2f}, "
      f"non-null: {train[PRIMARY_TARGET].notna().sum()}")

print(f"\n=== Test set ===")
print(f"Year range: {test['year'].min()}–{test['year'].max()}")
print(f"{PRIMARY_TARGET} — mean: {test[PRIMARY_TARGET].mean():.2f}, "
      f"median: {test[PRIMARY_TARGET].median():.2f}, "
      f"non-null: {test[PRIMARY_TARGET].notna().sum()}")

print(f"\n=== Feature stats (train) ===")
print(train[FEATURES].describe().round(3))

## Save train and test sets

In [ ]:
train.to_csv(DATA_PROCESSED / "train.csv", index=False)
test.to_csv(DATA_PROCESSED / "test.csv", index=False)

print(f"Saved: {DATA_PROCESSED / 'train.csv'} — {train.shape}")
print(f"Saved: {DATA_PROCESSED / 'test.csv'}  — {test.shape}")

## Note on scaling

Tree-based models (XGBoost, LightGBM, Random Forest) do **not** need feature scaling.  
For Ridge regression and MLP, we will apply `StandardScaler` in the training notebook,  
fit on training data only to avoid data leakage.

---
**Outputs produced:**
- `data/processed/train.csv` — 80% of data, all columns preserved
- `data/processed/test.csv` — 20% of data, all columns preserved

Both files contain features, all 3 poverty targets, metadata (country, year), and `imputation_pct`.

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.